In [1]:
import joblib
import shap
import numpy as np

# Load the winning model (XGBoost) and test data
xgb_model = joblib.load('../models/xgboost.pkl')
X_test = joblib.load('../data/processed/X_test.pkl')

# TreeExplainer is fast for XGBoost (vs KernelExplainer which is slow)
explainer = shap.TreeExplainer(xgb_model)

# Save the explainer so app loads it instantly
joblib.dump(explainer, '../models/shap_explainer.pkl')

# Also save feature names
feature_names = list(X_test.columns)
joblib.dump(feature_names, '../models/feature_names.pkl')

print(f"✅ SHAP explainer saved")
print(f"✅ Feature names saved: {len(feature_names)} features")
print(f"Features: {feature_names[:5]} ...")

✅ SHAP explainer saved
✅ Feature names saved: 30 features
Features: ['V1', 'V2', 'V3', 'V4', 'V5'] ...


In [2]:
# Quick test: explain the first fraud case in test set
y_test = joblib.load('../data/processed/y_test.pkl')
fraud_idx = np.where(y_test == 1)[0][0]
sample = X_test.iloc[[fraud_idx]]

shap_values = explainer.shap_values(sample)

# Top 5 features that pushed toward fraud
contributions = list(zip(feature_names, shap_values[0], sample.values[0]))
contributions.sort(key=lambda x: abs(x[1]), reverse=True)

print("🔍 TOP 5 FEATURES THAT EXPLAIN THIS FRAUD PREDICTION")
print("=" * 60)
for name, shap_val, feat_val in contributions[:5]:
    direction = "→ FRAUD" if shap_val > 0 else "→ LEGIT"
    print(f"  {name:8s} = {feat_val:+.2f}  (SHAP: {shap_val:+.3f} {direction})")

🔍 TOP 5 FEATURES THAT EXPLAIN THIS FRAUD PREDICTION
  V14      = -6.17  (SHAP: +4.411 → FRAUD)
  V10      = -4.88  (SHAP: +2.095 → FRAUD)
  V8       = +1.17  (SHAP: -1.213 → LEGIT)
  V12      = -4.69  (SHAP: +1.075 → FRAUD)
  V7       = -3.07  (SHAP: +0.943 → FRAUD)
